# SN-01bis — Sirenisation Phase 1 bis (rejeu adresse historique)

**Proposition** : quand la Phase 1 n'aboutit pas à un SIREN
« VALIDE_FORT », avant de retenir le simple « VALIDE » ou de passer en Phase 2,
on retente la validation du SIREN déjà lié en comparant l'adresse EJ FINESS
aux différentes adresses du siège disponibles dans SIRENE (adresses
historiques / alternatives).

**Pourquoi** : l'adresse de l'EJ dans FINESS n'a souvent pas été mise à jour,
alors que le siège du SIREN a déménagé dans SIRENE. L'ancienne adresse (celle de
FINESS) est en général encore présente dans SIRENE sous la forme d'un autre
établissement du même SIREN — souvent fermé (ancien siège).

**Méthode** : on reconstitue les adresses candidates à partir de tous les
établissements du SIREN (siège + secondaires + fermés), on rejoue le score
adresse, on garde le meilleur. Le score nom (indépendant de l'adresse) est
conservé de la Phase 1. On réutilise à l'identique `calc_score_adresse`,
`calc_score_global` et `classifier_resultat`.

**Objectif** : vérifier que le volume partant en Phase 2 diminue.

## 1. Imports

In [1]:
# Installation des packages
%pip install -r ../../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from pathlib import Path

from src.connexion  import get_duckdb_connection
from src.phase1bis  import (
    construire_adresses_alternatives, appliquer_phase1_bis,
)
from src.display    import afficher_tableau
from config.settings import (
    FINESS_EJ_CLEAN, SIRENE_UL_CLEAN, PARQUET_ETAB,
    RESULTS_SN_DIR, PROCESSED_DIR,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)

## 2. Rejouer la Phase 1

Relance de la Phase 1

In [3]:
from src.sirenisation import matching_direct_siren

df_ej = pd.read_parquet(FINESS_EJ_CLEAN)
df_ul = pd.read_parquet(SIRENE_UL_CLEAN)
df_ej['nmsiren_stru'] = df_ej['nmsiren_stru'].fillna('').astype(str)
df_ul['siren']        = df_ul['siren'].astype(str)

df_p1 = matching_direct_siren(df_ej, df_ul, desc='Phase 1 (rappel)')
print(f"Phase 1 : {len(df_p1):,} EJ")
print(df_p1['statut'].value_counts().to_string())

Phase 1 (rappel):   0%|          | 0/54185 [00:00<?, ?it/s]

Phase 1 : 54,185 EJ
statut
VALIDE_FORT      23059
VALIDE           15551
DOUTEUX           6712
SIREN_INCONNU     4103
REJETE            2660
SANS_SIREN        2100


## 3. Construire les adresses alternatives par SIREN

On charge tous les établissements SIRENE (actifs & fermés) via une vue
DuckDB dédiée, sans le filtre `etatAdministratifEtablissement = 'A'`

Pour limiter la mémoire, on ne garde que les SIREN présents dans la Phase 1.

In [4]:
# SIREN concernés (ceux liés en Phase 1)
sirens_utiles = set(df_p1['siren_ul'].dropna().astype(str).str.strip())
sirens_utiles.discard('')
sirens_utiles.discard('None')
print(f"SIREN à couvrir : {len(sirens_utiles):,}")

# Vue DuckDB : TOUS les établissements (actifs + fermés), colonnes d'adresse
con = get_duckdb_connection()
con.execute(f"""
    CREATE VIEW etab_all_adr AS
    SELECT siren,
           numeroVoieEtablissement, typeVoieEtablissement,
           libelleVoieEtablissement, codeCommuneEtablissement,
           etatAdministratifEtablissement, etablissementSiege
    FROM read_parquet('{PARQUET_ETAB}')
""")

# On récupère les établissements des SIREN utiles
df_sirens = pd.DataFrame({'siren': sorted(sirens_utiles)})
con.register('sirens_utiles', df_sirens)
df_etab_all = con.execute("""
    SELECT e.*
    FROM etab_all_adr e
    INNER JOIN sirens_utiles s ON e.siren = s.siren
""").df()
con.close()

print(f"Établissements récupérés : {len(df_etab_all):,}")
print("Répartition état :")
print(df_etab_all['etatAdministratifEtablissement'].value_counts().to_string())

SIREN à couvrir : 51,471


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Établissements récupérés : 291,959
Répartition état :
etatAdministratifEtablissement
A    171380
F    120579


In [5]:
# Normaliser les adresses alternatives (identique à pretraiter_ul)
df_adresses_alt = construire_adresses_alternatives(df_etab_all)
print(f"Adresses alternatives (lignes) : {len(df_adresses_alt):,}")
print(f"SIREN couverts                 : {df_adresses_alt['siren'].nunique():,}")

# Nombre moyen d'adresses par SIREN
nb_par_siren = df_adresses_alt.groupby('siren').size()
print(f"Adresses par SIREN — médiane : {nb_par_siren.median():.0f}, max : {nb_par_siren.max()}")
afficher_tableau(df_adresses_alt.head(10), 'Aperçu adresses alternatives')

Adresses alternatives (lignes) : 262,566
SIREN couverts                 : 51,414
Adresses par SIREN — médiane : 2, max : 13064


siren,numero_voie_norm_ul,libelle_voie_complet_ul,code_commune_norm_ul,etat,est_siege
100148261,1,RUE VIVARAIS,54547,A,True
100148261,2,RUE LORD BYRON,06029,A,False
100148261,7,ALLEE BUGADIERES,06027,A,False
100148261,168,AVENUE SEMPER OLIVA,83090,A,False
100148261,47,RUE HOTEL VILLE,83113,A,False


## 4. Appliquer la Phase 1 bis

On rejoue le score adresse pour tous les EJ dont la Phase 1 n'a pas donné
VALIDE_FORT. Le rejeu ne peut qu'améliorer ou
laisser inchangé (on garde le meilleur des deux : score P1 ou meilleure adresse
alternative).

In [6]:
df_bis = appliquer_phase1_bis(
    df_p1,
    df_adresses_alt,
    statuts_a_rejouer=('VALIDE', 'DOUTEUX', 'REJETE'),  
)

n_rejoues   = int(df_bis['phase1bis_applique'].sum())
n_ameliores = int(df_bis['phase1bis_gain'].notna().sum())
print(f"EJ rejoués           : {n_rejoues:,}")
print(f"EJ dont statut change: {n_ameliores:,}")

Phase 1 bis (rejeu adresse):   0%|          | 0/24923 [00:00<?, ?it/s]

EJ rejoués           : 24,923
EJ dont statut change: 7,014


## 5. Détail des gains de statut

In [7]:
gains = df_bis[df_bis['phase1bis_gain'].notna()]
print("Transitions de statut (Phase 1 → Phase 1 bis) :")
print(gains['phase1bis_gain'].value_counts().to_string())

print("\nExemples d'EJ récupérés (adresse alternative trouvée) :")
cols_apercu = ['nmsiren_stru', 'raisonsociale_norm_ej', 'statut', 'statut_bis',
               'score_adresse', 'score_adresse_bis', 'adresse_alt_retenue', 'etat_etab_retenu']
cols_apercu = [c for c in cols_apercu if c in gains.columns]
afficher_tableau(gains[cols_apercu].head(15), 'EJ récupérés par Phase 1 bis')

Transitions de statut (Phase 1 → Phase 1 bis) :
phase1bis_gain
VALIDE → VALIDE_FORT     2613
DOUTEUX → VALIDE_FORT    1498
DOUTEUX → VALIDE         1166
REJETE → VALIDE           800
REJETE → DOUTEUX          780
REJETE → VALIDE_FORT      157

Exemples d'EJ récupérés (adresse alternative trouvée) :


nmsiren_stru,raisonsociale_norm_ej,statut,statut_bis,score_adresse,score_adresse_bis,adresse_alt_retenue,etat_etab_retenu
261301733,C C S ROQUE ANTHERON,REJETE,VALIDE,37.680000,55.920000,RUE TEMPLE (13084),A
221300015,DIRECTION MAISONS ENFANCE,REJETE,VALIDE,16.290000,100.000000,12 RUE SAINT ADRIEN (13206),A
775557408,EDMOND BARTHELEMY,VALIDE,VALIDE_FORT,70.000000,100.000000,AVENUE VICTOR PEISSON (13087),A
775559727,ARAIMC,DOUTEUX,VALIDE_FORT,40.110000,90.740000,945 AVENUE PIC BERTAGNE (13042),F
775559891,INST REG SOURDS AVEUGLES MARSEILLE,DOUTEUX,VALIDE_FORT,37.740000,100.000000,1 RUE VAUVENARGUES (13207),A


## 5 bis. Comparaison adresses (FINESS / siège Phase 1 / alternative retenue)

Pour rendre le gain lisible, on rapproche trois adresses :
- adresse_finess_ej : l'adresse de l'EJ dans FINESS 
- adresse_siege_p1 : l'adresse du siège SIRENE utilisée en Phase 1 
- adresse_alt_retenue : l'adresse alternative du SIREN (retenue parmi tous les etab)

In [8]:
# Adresse FINESS de l'EJ (déjà construite par pretraiter_ej)
if "adresse_complete_ej" in df_bis.columns:
    df_bis["adresse_finess_ej"] = (
        df_bis["adresse_complete_ej"].fillna("").astype(str).str.strip()
        + " " + df_bis.get("cdcommune_stru", pd.Series("", index=df_bis.index)).fillna("").astype(str).str.strip()
    ).str.strip()
else:
    df_bis["adresse_finess_ej"] = (
        df_bis.get("nmvoie_stru", "").fillna("").astype(str).str.strip() + " "
        + df_bis.get("lbtypevoie_stru", "").fillna("").astype(str).str.strip() + " "
        + df_bis.get("lbvoie_stru", "").fillna("").astype(str).str.strip() + " "
        + df_bis.get("cdcommune_stru", "").fillna("").astype(str).str.strip()
    ).str.replace(r"\s+", " ", regex=True).str.strip()

# Adresse du siège utilisée en Phase 1 : jointe depuis df_ul via le SIREN
cols_siege = ["siren", "adresse_siege_complete_ul", "code_commune_norm_ul"]
cols_siege = [c for c in cols_siege if c in df_ul.columns]
ul_siege = df_ul[cols_siege].drop_duplicates("siren").copy()
ul_siege["siren"] = ul_siege["siren"].astype(str)

df_bis["siren_ul"] = df_bis["siren_ul"].astype(str)
df_bis = df_bis.merge(ul_siege, left_on="siren_ul", right_on="siren",
                      how="left", suffixes=("", "_siege")).drop(columns=["siren"], errors="ignore")

df_bis["adresse_siege_p1"] = (
    df_bis.get("adresse_siege_complete_ul", pd.Series("", index=df_bis.index)).fillna("").astype(str).str.strip()
    + " " + df_bis.get("code_commune_norm_ul", pd.Series("", index=df_bis.index)).fillna("").astype(str).str.strip()
).str.strip()

print("Adresses construites : adresse_finess_ej, adresse_siege_p1, adresse_alt_retenue")
apercu = df_bis[df_bis["phase1bis_gain"].notna()][
    ["nmsiren_stru", "adresse_finess_ej", "adresse_siege_p1", "adresse_alt_retenue"]
].head(10)
afficher_tableau(apercu, "Comparaison des adresses (EJ récupérés)")

Adresses construites : adresse_finess_ej, adresse_siege_p1, adresse_alt_retenue


nmsiren_stru,adresse_finess_ej,adresse_siege_p1,adresse_alt_retenue
261301733,HOTEL DE VILLE 13084,AV DU PIJORET 13084,RUE TEMPLE (13084)
221300015,12 R SAINT ADRIEN 13206,52 AVENUE DE SAINT JUST 13204,12 RUE SAINT ADRIEN (13206)
775557408,AV VICTOR PEISSON 13087,AVENUE VICTOR PEISSON 13087,AVENUE VICTOR PEISSON (13087)
775559727,945 AV DU PIC DE BRETAGNE 13042,CHEMIN DE FONT SEREINE 13042,945 AVENUE PIC BERTAGNE (13042)
775559891,1 R VAUVENARGUES 13207,8 MONTEE DE L'ORATOIRE 13207,1 RUE VAUVENARGUES (13207)


## 6. Mesure d'impact sur la Phase 2

Indicateur : le volume partant en Phase 2

In [9]:
def part_en_phase2(serie_statut):
    return ~serie_statut.isin(['VALIDE_FORT', 'VALIDE'])

n_p2_avant = int(part_en_phase2(df_bis['statut']).sum())
n_p2_apres = int(part_en_phase2(df_bis['statut_bis']).sum())
evites     = n_p2_avant - n_p2_apres

# Détail validés
valides_avant = int(df_bis['statut'].isin(['VALIDE_FORT', 'VALIDE']).sum())
valides_apres = int(df_bis['statut_bis'].isin(['VALIDE_FORT', 'VALIDE']).sum())
fort_avant    = int((df_bis['statut'] == 'VALIDE_FORT').sum())
fort_apres    = int((df_bis['statut_bis'] == 'VALIDE_FORT').sum())

print("=" * 58)
print("IMPACT PHASE 1 BIS")
print("=" * 58)
print(f"  VALIDE_FORT     : {fort_avant:>7,}  →  {fort_apres:>7,}   (+{fort_apres-fort_avant:,})")
print(f"  Validés (total) : {valides_avant:>7,}  →  {valides_apres:>7,}   (+{valides_apres-valides_avant:,})")
print("-" * 58)
print(f"  Partent en P2   : {n_p2_avant:>7,}  →  {n_p2_apres:>7,}   (-{evites:,})")
print("=" * 58)
if n_p2_avant > 0:
    print(f"  Réduction du volume Phase 2 : {evites/n_p2_avant*100:.1f} %")

IMPACT PHASE 1 BIS
  VALIDE_FORT     :  23,059  →   27,327   (+4,268)
  Validés (total) :  38,610  →   42,231   (+3,621)
----------------------------------------------------------
  Partent en P2   :  15,575  →   11,954   (-3,621)
  Réduction du volume Phase 2 : 23.2 %


## 7. Export Excel

In [10]:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment

SN_PHASE1BIS = RESULTS_SN_DIR / "sirenisation_phase1bis.xlsx"

# ─── Helpers de style (repris de la charte ANS de excel_export.py) ───────────
def _style_header(ws, couleur):
    fill = PatternFill("solid", start_color=couleur, end_color=couleur)
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

def _style_lignes(ws, couleur_alt):
    fa = PatternFill("solid", start_color=couleur_alt, end_color=couleur_alt)
    fb = PatternFill("solid", start_color="FFFFFF",    end_color="FFFFFF")
    for i, row in enumerate(ws.iter_rows(min_row=2), start=1):
        fill = fa if i % 2 == 0 else fb
        for cell in row:
            cell.fill      = fill
            cell.alignment = Alignment(horizontal="left", vertical="center")

def _auto_width(ws, max_w=45):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 4, max_w)

# Couleurs ANS
C_HEADER_SYNTH = "1F4E79"   # bleu marine
C_HEADER_GAIN  = "1A7341"   # vert (récupérés)
C_LIGNES_GAIN  = "E8F5E9"
C_HEADER_FULL  = "1F4E79"
C_LIGNES_FULL  = "F2F6FC"

# ─── Colonnes exportées ──────────────────────────────────────────────────────
COLS_EXPORT = [
    "idstructure_stru", "nmfinessej_stru", "dept_ej", "raisonsociale_stru",
    "nom_ul_retenu", "nmsiren_stru", "siren_ul",
    "score_nom", "score_adresse", "score_global", "statut",
    "score_adresse_bis", "score_global_bis", "statut_bis",
    "adresse_finess_ej", "adresse_siege_p1", "adresse_alt_retenue", 
    "etat_etab_retenu", "phase1bis_applique", "phase1bis_gain",
]
cols = [c for c in COLS_EXPORT if c in df_bis.columns]

# ─── Feuille 1 : Synthèse d'impact ───────────────────────────────────────────
df_synth = pd.DataFrame([
    {"Indicateur": "VALIDE_FORT avant",         "Nombre": fort_avant},
    {"Indicateur": "VALIDE_FORT après",         "Nombre": fort_apres},
    {"Indicateur": "Validés total avant",       "Nombre": valides_avant},
    {"Indicateur": "Validés total après",       "Nombre": valides_apres},
    {"Indicateur": "—",                         "Nombre": ""},
    {"Indicateur": "Partent en Phase 2 avant",  "Nombre": n_p2_avant},
    {"Indicateur": "Partent en Phase 2 après",  "Nombre": n_p2_apres},
    {"Indicateur": "EJ récupérés (P2 évités)",  "Nombre": evites},
])

# ─── Feuilles 2 et 3 ─────────────────────────────────────────────────────────
df_recuperes = df_bis[df_bis["phase1bis_gain"].notna()][cols]
df_complet   = df_bis[cols]

with pd.ExcelWriter(SN_PHASE1BIS, engine="openpyxl") as writer:
    # Feuille Synthèse
    df_synth.to_excel(writer, sheet_name="Synthese_impact", index=False)
    ws = writer.sheets["Synthese_impact"]
    _style_header(ws, C_HEADER_SYNTH)
    for row in ws.iter_rows(min_row=2):
        label = str(row[0].value)
        for cell in row:
            cell.alignment = Alignment(horizontal="left", vertical="center")
            if "évités" in label or "récupérés" in label:
                cell.font = Font(bold=True, name="Arial", size=10)
    _auto_width(ws, max_w=35)
    ws.freeze_panes = "A2"

    # Feuille EJ récupérés (vert)
    if len(df_recuperes) > 0:
        df_recuperes.to_excel(writer, sheet_name="EJ_recuperes", index=False)
        ws = writer.sheets["EJ_recuperes"]
        _style_header(ws, C_HEADER_GAIN)
        _style_lignes(ws, C_LIGNES_GAIN)
        _auto_width(ws)
        ws.freeze_panes = "A2"

    # Feuille résultat complet (bleu)
    df_complet.to_excel(writer, sheet_name="Resultat_complet", index=False)
    ws = writer.sheets["Resultat_complet"]
    _style_header(ws, C_HEADER_FULL)
    _style_lignes(ws, C_LIGNES_FULL)
    _auto_width(ws)
    ws.freeze_panes = "A2"

print(f"Export OK → {SN_PHASE1BIS.resolve()}")
print(f"  Synthese_impact  : réduction Phase 2 = {evites:,}")
print(f"  EJ_recuperes     : {len(df_recuperes):,} lignes")
print(f"  Resultat_complet : {len(df_complet):,} lignes")

Export OK → /home/jovyan/work/projet_finess_sirene/results/sirenisation/sirenisation_phase1bis.xlsx
  Synthese_impact  : réduction Phase 2 = 3,621
  EJ_recuperes     : 7,014 lignes
  Resultat_complet : 54,185 lignes
